# Outlier98 Behavioural Data Analysis and Model v3 Evaluation

This notebook presents the Data Science analysis undertaken for Outlier98, a data-driven productivity and behavioural analytics platform.

Two complementary evidence sources are considered. A controlled simulated behavioural dataset is used for reproducible model development and held-out evaluation, while anonymised real-user pilot aggregates are considered separately to examine practical data capture and feature availability.

The two sources answer different questions. Predictive-performance results come from the controlled modelling experiment; the pilot provides exploratory evidence about application use and longitudinal behavioural capture. The controlled experiment is synthetic in origin and does not establish real-world validity, causal effects or population-level generalisability.

## 1. Research context and analytical questions

This notebook addresses five analytical questions: whether recent behavioural variables can be organised into a consistent daily dataset; whether the engineered features contain predictive information in the controlled modelling dataset; how Logistic Regression compares with reasonable non-linear alternatives; how well Model v3 discriminates and calibrates on held-out users; and what the small pilot reveals about practical longitudinal data availability.

## 2. Data sources and evidence strategy

Predictive development requires multiple historical behavioural windows for each profile. The available three-participant pilot is small and uneven, with some participants contributing little or no longitudinal activity for several workflows. It is therefore unsuitable for model training or statistical validation. The controlled simulated behavioural dataset supplies reproducible user-day observations under defined assumptions, while the pilot is retained as separate evidence of practical data capture, workflow use and feature availability.

## 3. Target definition and unit of analysis

The unit of analysis is one controlled user-day. The target is `next_day_low_execution`. For prediction day D, the execution ratio is completed eligible planned activity divided by eligible planned activity. Planned activity consists of active Habits, due Tasks and scheduled T-Blocks. Completed activity consists of Done Habits, Partial Habits weighted as 0.5, completed due Tasks and completed T-Blocks. The target is 1 where the ratio is below 0.50 and 0 otherwise; days with no planned activity receive no target label. The 0.50 threshold is an operational definition for this prototype, not a universal definition of productivity.

Features use the seven complete local days D−7 to D−1 so that a recent behavioural history is available before the outcome day. Prediction-day information is excluded to prevent leakage from the behaviour the model is intended to predict.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score, recall_score,
                             f1_score, log_loss, brier_score_loss, confusion_matrix, roc_curve,
                             precision_recall_curve, average_precision_score)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

pd.set_option('display.max_columns', None)

# Works from the notebook directory, repository root, or a Colab checkout.
candidates = [
    Path('../data/model_v3_synthetic_dataset.csv'),
    Path('data/model_v3_synthetic_dataset.csv'),
    Path('/content/final_outlier98_thesis/data/model_v3_synthetic_dataset.csv'),
]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Place the repository in the working directory or update data_path.')

df = pd.read_csv(data_path)
print(f'Loaded: {data_path}')
print('Dataset shape:', df.shape)
print('Columns:', list(df.columns))
df.head()

## 4. Dataset checks

The verified experiment contains 31,800 labelled observations from 600 synthetic users. The split is by synthetic user, not by row.

In [ ]:
assert len(df) == 31800
assert df['synthetic_user_id'].nunique() == 600
assert (df['split'] == 'train').sum() == 25440
assert (df['split'] == 'held_out').sum() == 6360
assert df.loc[df['split'] == 'train', 'synthetic_user_id'].nunique() == 480
assert df.loc[df['split'] == 'held_out', 'synthetic_user_id'].nunique() == 120

print(df.groupby('split').agg(rows=('synthetic_user_id', 'size'), users=('synthetic_user_id', 'nunique')))
print('\nMissing values by column:')
print(df.isna().sum())

## 5. Target distribution

The target is 1 for low execution and 0 otherwise. The majority-class baseline is the larger of the positive and negative class proportions.

In [ ]:
held = df[df['split'] == 'held_out'].copy()
target_rate = held['next_day_low_execution'].mean()
majority_baseline = max(target_rate, 1 - target_rate)
print(f'Held-out positive rate: {target_rate:.6f}')
print(f'Held-out majority-class baseline: {majority_baseline:.6f}')
assert round(majority_baseline, 3) == 0.610
held['next_day_low_execution'].value_counts().sort_index().plot(kind='bar', rot=0, title='Held-out target distribution')
plt.xlabel('next_day_low_execution')
plt.ylabel('Observations')
plt.show()

The held-out set contains more non-low-execution than low-execution observations, which is why the majority-class baseline is approximately 0.610.

## 6. Feature overview

The model uses nine features. `habit_completion_rate_7d` uses Done Habit-days in its numerator; Partial is not equivalent to Done for this feature. Partial contributes 0.5 only in target construction. Recorded focus duration is used rather than planned T-Block duration. Missing values are not behavioural zeros.

In [ ]:
features = [
    'habit_completion_rate_7d', 'task_completion_rate_7d', 'tblock_completion_rate_7d',
    'avg_focus_mins_7d', 'current_streak', 'days_since_last_active',
    'reminder_usage_rate', 'avg_mood_7d', 'is_weekend'
]
display(df[features].describe().T)
display(df[features].isna().sum().rename('missing_values').to_frame())
df[features].hist(figsize=(14, 10), bins=25)
plt.tight_layout()
plt.show()

Missing values are handled using the training-set means rather than being interpreted as behavioural zeros.

## 7. Train / held-out split

Splitting by generated behavioural profile rather than by individual user-day reduces leakage from repeated and overlapping histories of the same profile. No generated user appears in both subsets.

In [ ]:
train = df[df['split'] == 'train'].copy()
train_users = set(train['synthetic_user_id'])
held_users = set(held['synthetic_user_id'])
assert train_users.isdisjoint(held_users)
print(f'Training users: {len(train_users)}; held-out users: {len(held_users)}')
print(f'Training rows: {len(train)}; held-out rows: {len(held)}')

## 8. Preprocessing

Training-set means and population standard deviations are used for standardisation. Missing numerical values are replaced by frozen training means. Prediction-day outcomes are excluded from historical features.

In [ ]:
means = train[features].mean(skipna=True)
scales = train[features].std(ddof=0, skipna=True).replace(0, 1)
X_train = ((train[features].fillna(means) - means) / scales).to_numpy(dtype=float)
X_held = ((held[features].fillna(means) - means) / scales).to_numpy(dtype=float)
y_train = train['next_day_low_execution'].to_numpy(dtype=float)
y_held = held['next_day_low_execution'].to_numpy(dtype=int)
display(pd.DataFrame({'mean': means, 'population_std': scales}))

## 9. Model v3

Model v3 uses binary Logistic Regression. It estimates the probability of low execution as the sigmoid of an intercept plus the weighted sum of standardised features. Standardisation places inputs on comparable scales before fitting. It is trained with full-batch gradient descent on L2-regularised log loss, using a learning rate of 0.5, 1,500 iterations, L2 regularisation of 0.001 and seed 20261007. For this public notebook, the calculation is reproduced directly in NumPy so that the analysis can run independently of the private application code.

In [ ]:
learning_rate = 0.5
iterations = 1500
l2 = 0.001
seed = 20261007

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

weights = np.zeros(X_train.shape[1], dtype=float)
intercept = 0.0
for _ in range(iterations):
    probabilities = sigmoid(intercept + X_train @ weights)
    error = probabilities - y_train
    gradient_w = (X_train.T @ error) / len(X_train) + l2 * weights
    gradient_b = error.mean()
    weights -= learning_rate * gradient_w
    intercept -= learning_rate * gradient_b

print('Intercept:', round(intercept, 10))
display(pd.DataFrame({'feature': features, 'coefficient': weights}))

### Training-only convergence diagnostic

The learning-rate diagnostic uses the training objective only. It is not a held-out hyperparameter search and does not change Model v3. The purpose is to check that the selected learning rate of 0.5 produces stable loss behaviour under a fixed, reproducible optimisation budget of 1,500 iterations. A small set of rates is shown to demonstrate optimisation behaviour rather than to claim that 0.5 is universally optimal.

In [ ]:
def training_objective(probability, target, coefficient, l2_value):
    clipped = np.clip(probability, 1e-12, 1 - 1e-12)
    data_loss = -np.mean(target * np.log(clipped) + (1 - target) * np.log(1 - clipped))
    return data_loss + 0.5 * l2_value * np.sum(coefficient ** 2)

def training_loss_trace(rate, l2_value=0.001, steps=1500):
    diagnostic_weights = np.zeros(X_train.shape[1])
    diagnostic_intercept = 0.0
    trace = []
    for step in range(1, steps + 1):
        probability = sigmoid(diagnostic_intercept + X_train @ diagnostic_weights)
        error = probability - y_train
        diagnostic_weights -= rate * ((X_train.T @ error) / len(X_train) + l2_value * diagnostic_weights)
        diagnostic_intercept -= rate * error.mean()
        if step == 1 or step % 25 == 0:
            updated_probability = sigmoid(diagnostic_intercept + X_train @ diagnostic_weights)
            trace.append((step, training_objective(updated_probability, y_train, diagnostic_weights, l2_value)))
    return pd.DataFrame(trace, columns=['iteration', 'training_objective'])

learning_rates = [0.10, 0.25, 0.50, 0.75]
loss_traces = {rate: training_loss_trace(rate) for rate in learning_rates}
plt.figure(figsize=(7, 4))
for rate, trace in loss_traces.items():
    plt.plot(trace['iteration'], trace['training_objective'], label=f'learning rate = {rate:.2f}')
plt.title('Training-objective convergence diagnostic')
plt.xlabel('Iteration')
plt.ylabel('L2-regularised training objective')
plt.title('Precision-recall curve: held-out controlled evaluation')
plt.legend()
plt.tight_layout()
plt.show()

convergence_summary = pd.DataFrame([{
    'learning_rate': rate,
    'final_training_objective': trace['training_objective'].iloc[-1],
    'change_over_final_100_iterations': trace['training_objective'].iloc[-1] - trace.loc[trace['iteration'] == 1400, 'training_objective'].iloc[0]
} for rate, trace in loss_traces.items()])
display(convergence_summary)
assert np.isfinite(convergence_summary[['final_training_objective', 'change_over_final_100_iterations']].to_numpy()).all()

The selected learning rate of 0.5 gives a stable decreasing training objective under the verified optimisation procedure. The final 100-iteration change provides a simple check that the objective has stabilised sufficiently within the fixed 1,500-iteration budget. This diagnostic supports reproducibility and optimisation stability; it does not use held-out results to tune Model v3.

## 10. Held-out predictions and classification metrics

In [ ]:
held_probability = sigmoid(intercept + X_held @ weights)
held_prediction = (held_probability >= 0.5).astype(int)
metrics = {
    'ROC-AUC': roc_auc_score(y_held, held_probability),
    'Accuracy': accuracy_score(y_held, held_prediction),
    'Precision': precision_score(y_held, held_prediction),
    'Recall': recall_score(y_held, held_prediction),
    'F1': f1_score(y_held, held_prediction),
    'Log loss': log_loss(y_held, held_probability),
    'Brier score': brier_score_loss(y_held, held_probability),
}
metrics_table = pd.DataFrame.from_dict(metrics, orient='index', columns=['value'])
display(metrics_table)
assert round(metrics['ROC-AUC'], 3) == 0.878
assert round(metrics['Accuracy'], 3) == 0.803
assert round(metrics['Precision'], 3) == 0.760
assert round(metrics['Recall'], 3) == 0.722
assert round(metrics['F1'], 3) == 0.740
assert round(metrics['Log loss'], 3) == 0.433
assert round(metrics['Brier score'], 3) == 0.138

Accuracy is not interpreted alone because the held-out majority-class baseline is approximately 0.610. ROC-AUC assesses ranking, precision and recall describe the positive low-execution class, F1 balances those measures, and log loss and Brier score assess probability quality. On held-out controlled users, Model v3 exceeds the baseline and shows useful discrimination, although these results remain specific to the controlled evaluation setting.

## 11. Confusion matrix

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_held, held_prediction).ravel()
confusion = pd.DataFrame(
    [[tp, fn], [fp, tn]],
    index=['Actual low execution', 'Actual not low execution'],
    columns=['Predicted low execution', 'Predicted not low execution']
)
display(confusion)
assert (tp, fp, tn, fn) == (1789, 566, 3316, 689)

The matrix shows that the model identifies many low-execution observations while still producing both false-positive and false-negative predictions.

## 12. ROC analysis

In [ ]:
fpr, tpr, _ = roc_curve(y_held, held_probability)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'ROC-AUC = {metrics["ROC-AUC"]:.3f}')
plt.plot([0, 1], [0, 1], '--', color='grey', label='Chance')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('Held-out synthetic ROC curve')
plt.legend()
plt.show()

The ROC curve shows discrimination above chance for the held-out synthetic observations.

## 13. Calibration analysis

Risk bands are Low for probabilities below 0.40, Medium for 0.40 to below 0.70, and High for probabilities of 0.70 or above.

In [ ]:
band = pd.cut(held_probability, bins=[-np.inf, 0.4, 0.7, np.inf], labels=['low', 'medium', 'high'], right=False)
calibration = pd.DataFrame({'risk_band': band, 'probability': held_probability, 'outcome': y_held}).groupby('risk_band', observed=False).agg(
    samples=('outcome', 'size'),
    mean_predicted=('probability', 'mean'),
    observed_low_execution_rate=('outcome', 'mean')
)
display(calibration)
assert list(calibration['samples']) == [3674, 1138, 1548]
assert [round(value, 4) for value in calibration['mean_predicted']] == [0.1366, 0.5620, 0.8188]
assert [round(value, 4) for value in calibration['observed_low_execution_rate']] == [0.1418, 0.5791, 0.8385]
calibration[['mean_predicted', 'observed_low_execution_rate']].plot(kind='bar', rot=0, title='Held-out synthetic calibration by risk band')
plt.ylabel('Probability / observed rate')
plt.show()

Within the synthetic held-out set, the observed low-execution rates are close to the mean predicted probabilities across the three risk bands.

## 14. Model interpretation

For a non-imputed feature, the linear explanation contribution is `coefficient × standardised feature`. This describes how an input affects the fitted model score. SHAP is not used. Coefficient contributions are not causal effects.

## 15. Results summary

The following table summarises the reproduced held-out synthetic results.

In [ ]:
summary = pd.DataFrame({
    'metric': ['ROC-AUC', 'Accuracy', 'Precision', 'Recall', 'F1', 'Log loss', 'Brier score'],
    'value': [metrics['ROC-AUC'], metrics['Accuracy'], metrics['Precision'], metrics['Recall'], metrics['F1'], metrics['Log loss'], metrics['Brier score']]
})
display(summary)

## 16. Limitations

Training data and held-out evaluation data are synthetic. The synthetic generator introduces structural circularity because outcomes are partly generated through related behavioural assumptions. The results show technical pipeline behaviour, not real-world predictive validity, causality or population-level generalisability.

The exploratory three-user real-user evaluation is separate from this synthetic experiment and is not statistical validation of Model v3.

## 17. Data quality audit

Before modelling, the controlled dataset is checked for structural problems that could invalidate the analysis. The checks cover dimensions, split overlap, duplicate user-day records, missingness, finite values, feature ranges, binary fields and date order.

In [ ]:
numeric_values = df[features].to_numpy(dtype=float)
observed_finite = np.isfinite(numeric_values[~np.isnan(numeric_values)]).all()
quality_summary = pd.DataFrame({
    'check': ['Rows', 'Unique controlled users', 'Duplicate user-day rows', 'Train/held-out user overlap', 'Finite observed feature values', 'Valid binary target', 'Valid weekend indicator', 'Chronological dates within each user'],
    'result': [len(df), df['synthetic_user_id'].nunique(), df.duplicated(['synthetic_user_id', 'local_day']).sum(), len(train_users & held_users), observed_finite, set(df['next_day_low_execution']).issubset({0, 1}), set(df['is_weekend']).issubset({0, 1}), all(group['local_day'].is_monotonic_increasing for _, group in df.groupby('synthetic_user_id'))]
})
display(quality_summary)
assert quality_summary.loc[2:3, 'result'].tolist() == [0, 0]
assert observed_finite

range_summary = pd.DataFrame({'minimum': df[features].min(), 'maximum': df[features].max(), 'missing': df[features].isna().sum()})
display(range_summary)

The audit finds one unique observation per controlled user-day and no split overlap. Missing values occur only where a feature is undefined for the available behavioural history; observed numerical values are finite. These checks support reproducible analysis, but do not remove the external-validity limitations of controlled simulated data.

## 18. Feature dictionary and missing-data analysis

The table below records the operational meaning of each feature. Associations in this notebook are descriptive rather than causal.

| Feature | Meaning and window | Missing condition | Conceptual direction |
|---|---|---|---|
| `habit_completion_rate_7d` | Done Habit-days / active Habit-days, D−7 to D−1 | No active Habits | Higher recent completion may indicate lower risk |
| `task_completion_rate_7d` | Historically valid completed eligible due Tasks / eligible due Tasks, D−7 to D−1 | No eligible due Tasks | Higher recent completion may indicate lower risk |
| `tblock_completion_rate_7d` | Completed eligible T-Blocks / scheduled eligible T-Blocks, D−7 to D−1 | No scheduled eligible T-Blocks | Higher recent completion may indicate lower risk |
| `avg_focus_mins_7d` | Average recorded focus-session duration, D−7 to D−1 | Not missing in this controlled dataset | Higher recorded focus may indicate lower risk |
| `current_streak` | Qualifying Habit streak ending before D | Zero where no qualifying prior streak exists | Longer continuity may indicate lower risk |
| `days_since_last_active` | Days from D to latest prior activity, capped at 14 | Not missing after capping | More elapsed days may indicate higher risk |
| `reminder_usage_rate` | Created Tasks with reminders / created Tasks, D−7 to D−1 | No Task creation opportunity | Direction is empirical, not assumed |
| `avg_mood_7d` | Mean available mood observations, D−7 to D−1 | No mood observations | Direction is empirical, not assumed |
| `is_weekend` | Whether prediction day D is Saturday or Sunday | Never missing | Calendar context only |

Partial Habit completion is not equivalent to Done for `habit_completion_rate_7d`; it contributes 0.5 only when constructing the target. Recorded focus duration is not replaced with planned T-Block duration.

### Feature inclusion and exclusion rationale

The nine inputs were retained because each describes recent execution, recency, available behavioural context or calendar context using a defined pre-prediction window. Variables were excluded where their operational definition would weaken the distinction between planned activity, observed execution and historically defensible evidence.

| Variable not used as a Model v3 feature | Rationale |
|---|---|
| Planned T-Block duration as focus | Planned duration represents intention or scheduling, not recorded focus-session duration; treating it as observed focus would confuse planned work with recorded execution. |
| D-SIREN longitudinal distraction measures | A compatible, verified longitudinal distraction series is not part of the controlled modelling dataset, so it cannot be used consistently across the model-development observations. |
| XP and gamification scores | These are derived gamification summaries rather than direct measures of recent planned and recorded execution, and could introduce a redundant reward signal. |
| Weak or legacy analytical measures | Measures without a stable historical definition are excluded to avoid combining non-comparable evidence across time. |
| Mission progress | This is derived from linked Task-plan status and is not used as an independent predictive feature, avoiding duplication or obscuring the underlying Task-execution signal. |
| Current Task status without historical reconstruction | Present status alone is not historical proof that a Task was completed by the relevant due day; Model v3 instead requires historically valid completion evidence. |

These exclusions are modelling choices, not claims that the excluded variables are unimportant in every context.

In [ ]:
missing_counts = df[features].isna().sum().sort_values(ascending=False)
missing_counts.plot(kind='bar', figsize=(9, 4), color='steelblue', title='Feature availability in the controlled modelling dataset')
plt.ylabel('Missing observations')
plt.xlabel('Feature')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

# Mean imputation makes a missing numerical input neutral after standardisation: z = (mean - mean) / scale = 0.
imputation_check = pd.DataFrame({'training_mean': means, 'standardised_when_imputed': ((means - means) / scales)})
display(imputation_check)
assert np.allclose(imputation_check['standardised_when_imputed'], 0)

Mean imputation supplies a complete numerical vector without treating unavailable behaviour as zero. After standardisation, an imputed value is zero and therefore has no linear feature contribution. This is a practical scoring rule, not recovery of missing behavioural information.

## 19. Exploratory relationships

Spearman correlation is used because the dataset combines rates, bounded measurements, counts and binary variables. Correlation measures association only; it does not establish causal behavioural relationships.

In [ ]:
correlation_data = df[features + ['next_day_low_execution']]
spearman = correlation_data.corr(method='spearman')
plt.figure(figsize=(10, 8))
image = plt.imshow(spearman, vmin=-1, vmax=1, cmap='coolwarm')
plt.colorbar(image, label='Spearman correlation')
plt.xticks(range(len(spearman.columns)), spearman.columns, rotation=75, ha='right')
plt.yticks(range(len(spearman.index)), spearman.index)
plt.title('Feature associations in the controlled modelling dataset')
plt.xlabel('Feature or target')
plt.ylabel('Feature or target')
plt.tight_layout()
plt.show()
display(spearman['next_day_low_execution'].sort_values().to_frame('Spearman correlation with target'))

The correlation view is used to inspect the structure available to the model and potential redundancy between engineered inputs. It should not be read as evidence that changing one behavioural measure causes a change in next-day execution.

## 20. Exploratory candidate-model comparison

The comparison uses the same held-out users for all candidates. Configurations are fixed before evaluation and are not tuned against the held-out set. This is an exploratory analytical comparison, not definitive model selection, and it does not alter the implemented Model v3 configuration.

In [ ]:
candidate_models = {
    'Dummy prior': Pipeline([('impute', SimpleImputer(strategy='mean')), ('model', DummyClassifier(strategy='prior'))]),
    'scikit-learn Logistic Regression candidate': Pipeline([('impute', SimpleImputer(strategy='mean')), ('scale', StandardScaler()), ('model', LogisticRegression(C=1000, max_iter=2000, random_state=20261007))]),
    'Random Forest': Pipeline([('impute', SimpleImputer(strategy='mean')), ('model', RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=20261007, n_jobs=1))]),
    'Histogram Gradient Boosting': Pipeline([('impute', SimpleImputer(strategy='mean')), ('model', HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15, l2_regularization=0.001, random_state=20261007))]),
}
comparison_rows = []
for name, candidate in candidate_models.items():
    candidate.fit(train[features], y_train)
    probability = candidate.predict_proba(held[features])[:, 1]
    prediction = (probability >= 0.5).astype(int)
    comparison_rows.append({'model': name, 'ROC-AUC': roc_auc_score(y_held, probability), 'Accuracy': accuracy_score(y_held, prediction), 'Precision': precision_score(y_held, prediction, zero_division=0), 'Recall': recall_score(y_held, prediction, zero_division=0), 'F1': f1_score(y_held, prediction, zero_division=0), 'Log loss': log_loss(y_held, probability, labels=[0, 1]), 'Brier': brier_score_loss(y_held, probability)})
comparison = pd.DataFrame(comparison_rows).set_index('model')
display(comparison.round(6))
comparison[['ROC-AUC', 'F1', 'Brier']].plot(kind='bar', figsize=(9, 4), rot=20, title='Candidate comparison on held-out users')
plt.xlabel('Candidate configuration')
plt.ylabel('Metric value')
plt.tight_layout()
plt.show()

Histogram Gradient Boosting has the strongest performance among the candidate configurations evaluated in this controlled comparison. Random Forest also slightly exceeds the separate scikit-learn Logistic Regression candidate on several measures. The observed gains over Logistic Regression are modest and do not resolve the external-validity limitation of the controlled dataset. The implemented full-batch-gradient-descent Logistic Regression Model v3 is retained because its performance is competitive while it supports deterministic scoring, direct coefficient-based explanations, low computational complexity and reproducible user-facing probability calculations. This is a performance–interpretability trade-off, not a claim that Logistic Regression is universally superior.

Deep learning is not added simply for model complexity. The analysis has nine engineered tabular features, and a more complex model would not resolve the lack of large-scale genuine longitudinal validation while reducing direct interpretability.

## 21. Precision–recall, threshold and regularisation sensitivity

ROC-AUC evaluates ranking across thresholds. Precision–recall analysis focuses more directly on the low-execution class. Threshold sensitivity is reported to show the warning trade-off, not to select a new production threshold. The implemented classification threshold remains 0.50; risk bands use separate 0.40 and 0.70 boundaries.

In [ ]:
precision_curve, recall_curve, _ = precision_recall_curve(y_held, held_probability)
average_precision = average_precision_score(y_held, held_probability)
plt.figure(figsize=(6, 5))
plt.plot(recall_curve, precision_curve, label=f'Average precision = {average_precision:.3f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision–recall curve: held-out controlled evaluation')
plt.legend()
plt.show()
print(f'Average precision: {average_precision:.6f}')

threshold_rows = []
for threshold in np.arange(0.30, 0.76, 0.05):
    predicted = (held_probability >= threshold).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_held, predicted).ravel()
    threshold_rows.append({'threshold': round(float(threshold), 2), 'precision': precision_score(y_held, predicted), 'recall': recall_score(y_held, predicted), 'f1': f1_score(y_held, predicted), 'false_positives': fp_t, 'false_negatives': fn_t})
threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table)
threshold_table.plot(x='threshold', y=['precision', 'recall', 'f1'], marker='o', figsize=(7, 4), title='Threshold sensitivity on held-out users')
plt.xlabel('Classification threshold')
plt.ylabel('Score')
plt.show()

In [ ]:
def fit_v3_with_l2(l2_value):
    sensitivity_weights = np.zeros(X_train.shape[1])
    sensitivity_intercept = 0.0
    for _ in range(1500):
        probability = sigmoid(sensitivity_intercept + X_train @ sensitivity_weights)
        error = probability - y_train
        sensitivity_weights -= 0.5 * ((X_train.T @ error) / len(X_train) + l2_value * sensitivity_weights)
        sensitivity_intercept -= 0.5 * error.mean()
    return sigmoid(sensitivity_intercept + X_held @ sensitivity_weights)

sensitivity_rows = []
for l2_value in [0, 0.0001, 0.001, 0.01, 0.1]:
    probability = fit_v3_with_l2(l2_value)
    predicted = (probability >= 0.5).astype(int)
    sensitivity_rows.append({'L2': l2_value, 'ROC-AUC': roc_auc_score(y_held, probability), 'F1': f1_score(y_held, predicted), 'Brier': brier_score_loss(y_held, probability)})
sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity.round(6))
sensitivity.set_index('L2').plot(marker='o', figsize=(7, 4), title='Regularisation sensitivity')
plt.xlabel('L2 regularisation value')
plt.ylabel('Metric value')
plt.show()

Lower thresholds increase sensitivity and warnings, but also increase false positives; higher thresholds reduce warnings while increasing false negatives. The L2 experiment is a sensitivity and robustness analysis, not held-out hyperparameter optimisation. It examines whether Model v3 depends heavily on one exact L2 value and is not used to replace the verified Model v3 setting of L2 = 0.001.

## 22. Coefficients, calibration and error interpretation

Because inputs are standardised, coefficient magnitudes are more comparable than values fitted on unrelated original scales. A positive coefficient increases the model logit for low execution, while a negative coefficient reduces it. Coefficients and feature contributions are not causal measures or proof of behavioural importance in real populations.

In [ ]:
coefficient_table = pd.DataFrame({'feature': features, 'coefficient': weights}).sort_values('coefficient')
display(coefficient_table)
plt.figure(figsize=(7, 4))
plt.barh(coefficient_table['feature'], coefficient_table['coefficient'], color='slateblue')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Standardised Logistic Regression coefficients')
plt.xlabel('Coefficient')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

calibration_plot = calibration.reset_index()
calibration_plot.plot(x='risk_band', y=['mean_predicted', 'observed_low_execution_rate'], kind='bar', rot=0, figsize=(7, 4), title='Probability calibration by risk band')
plt.xlabel('Risk band')
plt.ylabel('Probability / observed rate')
plt.tight_layout()
plt.show()

At the 0.50 classification threshold, false positives represent unnecessary low-execution warnings and false negatives represent missed opportunities to flag a difficult day. Calibration is also important because the system presents probability and risk information, not only binary labels. The displayed calibration is limited to the controlled held-out evaluation dataset.

## 23. Real-user exploratory behavioural analysis

The following privacy-safe aggregates are separate from model training and evaluation. They describe a small and uneven real-user evaluation dataset involving three anonymised participants. Their purpose is to assess data-capture feasibility, workflow use and feature availability rather than predictive accuracy.

In [ ]:
pilot = pd.DataFrame({
    'participant': ['P01', 'P02', 'P03'],
    'Tasks': [21, 6, 8], 'NIKA events': [218, 3, 8], 'Habits': [6, 4, 2],
    'Habit logs': [223, 0, 0], 'T-Blocks': [29, 0, 0], 'Focus sessions': [13, 0, 0],
    'Recorded focus minutes': [471, 0, 0], 'Stored predictions': [9, 1, 0]
})
display(pilot)
coverage = pilot.set_index('participant').gt(0).astype(int)
plt.figure(figsize=(9, 3))
plt.imshow(coverage, aspect='auto', cmap='Blues', vmin=0, vmax=1)
plt.xticks(range(len(coverage.columns)), coverage.columns, rotation=35, ha='right')
plt.yticks(range(len(coverage.index)), coverage.index)
plt.colorbar(label='Workflow evidence available')
plt.title('Real-user pilot workflow coverage')
plt.tight_layout()
plt.show()
pilot.plot(x='participant', y='Recorded focus minutes', kind='bar', rot=0, legend=False, title='Recorded focus minutes in the real-user pilot')
plt.xlabel('Participant')
plt.ylabel('Minutes')
plt.show()

P01 generated substantially richer longitudinal records across Tasks, Habit logs, T-Blocks, recorded focus and stored predictions. P02 and P03 used a smaller subset of workflows. This uneven engagement is relevant to feature availability: a predictive system needs sufficient historical records to construct reliable inputs. The pilot shows practical data-capture feasibility, but it does not support real-user ROC-AUC, F1, calibration, causal claims or population-level generalisation.

## 24. Evidence strategy, limitations and reproducibility

| Evidence source | Main purpose | Strength | Limitation |
|---|---|---|---|
| Controlled model-development dataset | Reproducible model development and evaluation | Large, repeatable user-level experiment | Limited external behavioural validity |
| Real-user evaluation dataset | Practical data capture and workflow coverage | Genuine application-use evidence | Small and uneven engagement |

The two streams are complementary rather than interchangeable. The notebook can be reproduced from the public controlled dataset with fixed settings and random states. More complex models cannot compensate for limited genuine longitudinal validation.

## 25. Final analytical conclusion

The controlled experiment shows that the engineered variables can be organised into a reproducible daily modelling dataset and that Model v3 provides competitive discrimination and probability estimates on held-out controlled users. Non-linear candidates provide a useful comparison, while the retained Logistic Regression offers transparent, deterministic scoring and direct feature-contribution tracing for the implemented system.

The real-user pilot separately shows that longitudinal behavioural capture is feasible but uneven. The next empirical step is prospective collection of a larger genuine longitudinal dataset with predictions stored before the outcome day.